# Comprehensive NLP Analysis and Modeling Pipeline

A complete natural language processing pipeline with text analysis, classification, entity recognition, topic modeling, sentiment analysis, and advanced NLP capabilities.

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import re
import string
from pathlib import Path
import json
import warnings
from typing import List, Dict, Optional, Tuple, Union, Any
from dataclasses import dataclass, field
from collections import Counter, defaultdict
import pickle
from datetime import datetime
import time

# NLP libraries
import spacy
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.sentiment import SentimentIntensityAnalyzer

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, HashingVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF, TruncatedSVD
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

# Deep Learning & Transformers
try:
    from transformers import (
        pipeline, 
        AutoTokenizer, 
        AutoModelForSequenceClassification,
        AutoModelForQuestionAnswering,
        AutoModelForTokenClassification,
        AutoModelForSeq2SeqLM,
        AutoModelForCausalLM
    )
    import torch
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    print("Transformers not installed. Run: pip install transformers torch")
    TRANSFORMERS_AVAILABLE = False

# Gensim
try:
    from gensim.models import Word2Vec, Doc2Vec, FastText, LdaModel, LdaMulticore
    from gensim.models.doc2vec import TaggedDocument
    from gensim.corpora import Dictionary
    from gensim.models.coherencemodel import CoherenceModel
    GENSIM_AVAILABLE = True
except ImportError:
    print("Gensim not installed. Run: pip install gensim")
    GENSIM_AVAILABLE = False

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Download required NLTK data
nltk_downloads = ['punkt', 'stopwords', 'wordnet', 'vader_lexicon', 'averaged_perceptron_tagger', 'maxent_ne_chunker', 'words']
for resource in nltk_downloads:
    try:
        nltk.download(resource, quiet=True)
    except:
        pass

print("NLP Pipeline initialized successfully!")
print(f"Transformers available: {TRANSFORMERS_AVAILABLE}")
print(f"Gensim available: {GENSIM_AVAILABLE}")

## 1. Comprehensive Text Analysis Pipeline

In [ ]:
@dataclass
class TextAnalysisConfig:
    """Configuration for text analysis."""
    language: str = 'en'
    spacy_model: str = 'en_core_web_sm'
    max_features: int = 5000
    ngram_range: Tuple[int, int] = (1, 3)
    min_df: int = 2
    max_df: float = 0.95
    use_lemmatization: bool = True
    use_stemming: bool = False
    remove_stopwords: bool = True
    lowercase: bool = True
    remove_punctuation: bool = True
    remove_numbers: bool = False
    min_word_length: int = 2


class ComprehensiveTextAnalyzer:
    """Complete text analysis pipeline."""
    
    def __init__(self, config: Optional[TextAnalysisConfig] = None):
        self.config = config or TextAnalysisConfig()
        
        # Load spaCy model
        try:
            self.nlp = spacy.load(self.config.spacy_model)
        except:
            print(f"Downloading {self.config.spacy_model}...")
            import subprocess
            subprocess.run(["python", "-m", "spacy", "download", self.config.spacy_model])
            self.nlp = spacy.load(self.config.spacy_model)
        
        # Initialize components
        self.lemmatizer = WordNetLemmatizer()
        self.stemmer = PorterStemmer()
        self.stop_words = set(stopwords.words(self.config.language))
        self.sia = SentimentIntensityAnalyzer()
        
        # Initialize transformers pipelines if available
        if TRANSFORMERS_AVAILABLE:
            self.sentiment_pipeline = pipeline('sentiment-analysis', device=-1)
            self.ner_pipeline = pipeline('ner', aggregation_strategy='simple', device=-1)
            self.summarization_pipeline = pipeline('summarization', device=-1)
            self.qa_pipeline = pipeline('question-answering', device=-1)
    
    def analyze_text(self, text: str, detailed: bool = True) -> Dict:
        """Comprehensive text analysis."""
        
        doc = self.nlp(text)
        
        analysis = {
            'basic_stats': self._basic_statistics(text, doc),
            'readability': self._readability_scores(text),
            'sentiment': self._sentiment_analysis(text),
            'entities': self._named_entity_recognition(doc),
            'key_phrases': self._extract_key_phrases(doc, text),
            'pos_distribution': self._pos_distribution(doc),
            'linguistic_features': self._linguistic_features(doc),
        }
        
        if detailed:
            analysis['dependency_patterns'] = self._dependency_patterns(doc)
            analysis['text_complexity'] = self._text_complexity(doc)
            analysis['stylistic_features'] = self._stylistic_features(text, doc)
        
        return analysis
    
    def _basic_statistics(self, text: str, doc) -> Dict:
        """Calculate basic text statistics."""
        
        sentences = list(doc.sents)
        words = [token for token in doc if not token.is_punct]
        unique_words = set([token.lower_ for token in words])
        
        return {
            'char_count': len(text),
            'char_count_no_spaces': len(text.replace(' ', '')),
            'word_count': len(words),
            'unique_word_count': len(unique_words),
            'sentence_count': len(sentences),
            'paragraph_count': len(text.split('\n\n')),
            'avg_word_length': np.mean([len(token.text) for token in words]) if words else 0,
            'avg_sentence_length': np.mean([len(list(sent)) for sent in sentences]) if sentences else 0,
            'vocabulary_richness': len(unique_words) / len(words) if words else 0,
            'lexical_diversity': len(unique_words) / np.sqrt(len(words)) if words else 0
        }
    
    def _readability_scores(self, text: str) -> Dict:
        """Calculate various readability scores."""
        
        try:
            import textstat
            
            return {
                'flesch_reading_ease': textstat.flesch_reading_ease(text),
                'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text),
                'gunning_fog': textstat.gunning_fog(text),
                'automated_readability_index': textstat.automated_readability_index(text),
                'coleman_liau_index': textstat.coleman_liau_index(text),
                'dale_chall': textstat.dale_chall_readability_score(text),
                'reading_time_minutes': textstat.reading_time(text, ms_per_char=14.69) / 60000
            }
        except ImportError:
            print("textstat not installed. Run: pip install textstat")
            return {}
    
    def _sentiment_analysis(self, text: str) -> Dict:
        """Perform sentiment analysis using multiple methods."""
        
        sentiments = {}
        
        # NLTK VADER sentiment
        vader_scores = self.sia.polarity_scores(text)
        sentiments['vader'] = vader_scores
        
        # Transformers sentiment (if available)
        if TRANSFORMERS_AVAILABLE and len(text) < 512:
            try:
                transformer_sentiment = self.sentiment_pipeline(text[:512])[0]
                sentiments['transformer'] = {
                    'label': transformer_sentiment['label'],
                    'score': transformer_sentiment['score']
                }
            except:
                pass
        
        # TextBlob sentiment (if available)
        try:
            from textblob import TextBlob
            blob = TextBlob(text)
            sentiments['textblob'] = {
                'polarity': blob.sentiment.polarity,
                'subjectivity': blob.sentiment.subjectivity
            }
        except ImportError:
            pass
        
        return sentiments
    
    def _named_entity_recognition(self, doc) -> Dict:
        """Extract named entities."""
        
        entities = defaultdict(list)
        
        # SpaCy NER
        for ent in doc.ents:
            entities[ent.label_].append(ent.text)
        
        # Remove duplicates and convert to regular dict
        entities = {k: list(set(v)) for k, v in entities.items()}
        
        # Add entity statistics
        entity_stats = {
            'total_entities': sum(len(v) for v in entities.values()),
            'entity_types': len(entities),
            'entities_by_type': {k: len(v) for k, v in entities.items()}
        }
        
        return {
            'entities': entities,
            'statistics': entity_stats
        }
    
    def _extract_key_phrases(self, doc, text: str) -> Dict:
        """Extract key phrases using multiple methods."""
        
        key_phrases = {}
        
        # Method 1: Noun phrases from spaCy
        noun_phrases = [chunk.text for chunk in doc.noun_chunks]
        key_phrases['noun_phrases'] = Counter(noun_phrases).most_common(10)
        
        # Method 2: RAKE algorithm (if available)
        try:
            from rake_nltk import Rake
            rake = Rake()
            rake.extract_keywords_from_text(text)
            key_phrases['rake'] = rake.get_ranked_phrases()[:10]
        except ImportError:
            pass
        
        # Method 3: YAKE keyword extraction (if available)
        try:
            import yake
            kw_extractor = yake.KeywordExtractor(
                lan=self.config.language,
                n=3,
                dedupLim=0.7,
                top=10
            )
            keywords = kw_extractor.extract_keywords(text)
            key_phrases['yake'] = [(kw[0], round(kw[1], 4)) for kw in keywords]
        except ImportError:
            pass
        
        # Method 4: TF-IDF based (single document)
        if len(text.split()) > 10:
            vectorizer = TfidfVectorizer(
                ngram_range=(1, 3),
                max_features=20,
                stop_words='english'
            )
            try:
                tfidf_matrix = vectorizer.fit_transform([text])
                feature_names = vectorizer.get_feature_names_out()
                scores = tfidf_matrix.toarray()[0]
                top_indices = scores.argsort()[-10:][::-1]
                key_phrases['tfidf'] = [(feature_names[i], round(scores[i], 4)) for i in top_indices]
            except:
                pass
        
        return key_phrases
    
    def _pos_distribution(self, doc) -> Dict:
        """Analyze part-of-speech distribution."""
        
        pos_counts = Counter([token.pos_ for token in doc])
        total = sum(pos_counts.values())
        
        return {
            'counts': dict(pos_counts),
            'percentages': {pos: round(count/total * 100, 2) 
                          for pos, count in pos_counts.items()},
            'most_common': pos_counts.most_common(5)
        }
    
    def _linguistic_features(self, doc) -> Dict:
        """Extract linguistic features."""
        
        # Count specific patterns
        passive_voice = sum(1 for token in doc if token.dep_ == 'nsubjpass')
        questions = sum(1 for sent in doc.sents if sent.text.strip().endswith('?'))
        exclamations = sum(1 for sent in doc.sents if sent.text.strip().endswith('!'))
        
        # Modality markers
        modal_verbs = ['can', 'could', 'may', 'might', 'must', 'shall', 'should', 'will', 'would']
        modals = sum(1 for token in doc if token.text.lower() in modal_verbs)
        
        # Complexity indicators
        subordinate_clauses = sum(1 for token in doc if token.dep_ == 'mark')
        conjunctions = sum(1 for token in doc if token.pos_ == 'CCONJ')
        
        return {
            'passive_voice_count': passive_voice,
            'question_count': questions,
            'exclamation_count': exclamations,
            'modal_verb_count': modals,
            'subordinate_clause_count': subordinate_clauses,
            'conjunction_count': conjunctions
        }
    
    def _dependency_patterns(self, doc) -> Dict:
        """Analyze dependency patterns."""
        
        dep_counts = Counter([token.dep_ for token in doc])
        
        # Extract subject-verb-object triples
        svo_triples = []
        for sent in doc.sents:
            for token in sent:
                if token.dep_ == 'ROOT':
                    subjects = [child for child in token.children if 'subj' in child.dep_]
                    objects = [child for child in token.children if 'obj' in child.dep_]
                    
                    for subj in subjects:
                        for obj in objects:
                            svo_triples.append((subj.text, token.text, obj.text))
        
        return {
            'dependency_counts': dict(dep_counts.most_common(10)),
            'svo_triples': svo_triples[:10]
        }
    
    def _text_complexity(self, doc) -> Dict:
        """Analyze text complexity."""
        
        # Syntactic complexity
        tree_depths = []
        for sent in doc.sents:
            depths = [0]
            for token in sent:
                depth = 0
                current = token
                while current.head != current:
                    depth += 1
                    current = current.head
                depths.append(depth)
            tree_depths.append(max(depths))
        
        # Vocabulary complexity
        word_lengths = [len(token.text) for token in doc if not token.is_punct]
        syllable_counts = self._estimate_syllables(doc)
        
        return {
            'max_tree_depth': max(tree_depths) if tree_depths else 0,
            'avg_tree_depth': np.mean(tree_depths) if tree_depths else 0,
            'long_word_ratio': sum(1 for w in word_lengths if w > 6) / len(word_lengths) if word_lengths else 0,
            'avg_syllables_per_word': np.mean(syllable_counts) if syllable_counts else 0
        }
    
    def _estimate_syllables(self, doc) -> List[int]:
        """Estimate syllable count for words."""
        
        def count_syllables(word):
            word = word.lower()
            vowels = 'aeiouAEIOU'
            syllable_count = 0
            previous_was_vowel = False
            
            for char in word:
                is_vowel = char in vowels
                if is_vowel and not previous_was_vowel:
                    syllable_count += 1
                previous_was_vowel = is_vowel
            
            if word.endswith('e'):
                syllable_count -= 1
            if syllable_count == 0:
                syllable_count = 1
            
            return syllable_count
        
        return [count_syllables(token.text) for token in doc if not token.is_punct]
    
    def _stylistic_features(self, text: str, doc) -> Dict:
        """Extract stylistic features."""
        
        # Punctuation usage
        punct_counts = Counter([token.text for token in doc if token.is_punct])
        
        # Capitalization patterns
        cap_words = sum(1 for token in doc if token.text[0].isupper() and not token.is_sent_start)
        all_caps = sum(1 for token in doc if token.text.isupper() and len(token.text) > 1)
        
        # Repetition patterns
        word_freq = Counter([token.lower_ for token in doc if not token.is_punct])
        repeated_words = sum(1 for count in word_freq.values() if count > 3)
        
        return {
            'punctuation_distribution': dict(punct_counts.most_common(5)),
            'capitalized_words': cap_words,
            'all_caps_words': all_caps,
            'highly_repeated_words': repeated_words,
            'unique_punctuation': len(punct_counts)
        }
    
    def preprocess_text(self, text: str) -> str:
        """Preprocess text based on configuration."""
        
        # Lowercase
        if self.config.lowercase:
            text = text.lower()
        
        # Remove punctuation
        if self.config.remove_punctuation:
            text = text.translate(str.maketrans('', '', string.punctuation))
        
        # Remove numbers
        if self.config.remove_numbers:
            text = re.sub(r'\d+', '', text)
        
        # Tokenize
        tokens = word_tokenize(text)
        
        # Remove stopwords
        if self.config.remove_stopwords:
            tokens = [t for t in tokens if t not in self.stop_words]
        
        # Minimum word length
        tokens = [t for t in tokens if len(t) >= self.config.min_word_length]
        
        # Lemmatization or stemming
        if self.config.use_lemmatization:
            tokens = [self.lemmatizer.lemmatize(t) for t in tokens]
        elif self.config.use_stemming:
            tokens = [self.stemmer.stem(t) for t in tokens]
        
        return ' '.join(tokens)

## 2. Advanced Topic Modeling

In [ ]:
class AdvancedTopicModeling:
    """Advanced topic modeling with multiple algorithms."""
    
    def __init__(self, n_topics: int = 10, random_state: int = 42):
        self.n_topics = n_topics
        self.random_state = random_state
        self.models = {}
        self.vectorizer = None
        self.doc_term_matrix = None
        
    def fit_all_models(self, documents: List[str], 
                      preprocess: bool = True) -> Dict:
        """Fit multiple topic models."""
        
        print("Preprocessing documents...")
        if preprocess:
            analyzer = ComprehensiveTextAnalyzer()
            documents = [analyzer.preprocess_text(doc) for doc in documents]
        
        # Create document-term matrix
        print("Creating document-term matrix...")
        self.vectorizer = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            max_df=0.95,
            min_df=2
        )
        self.doc_term_matrix = self.vectorizer.fit_transform(documents)
        
        results = {}
        
        # Fit different models
        print("Fitting LDA...")
        results['lda'] = self._fit_lda(documents)
        
        print("Fitting NMF...")
        results['nmf'] = self._fit_nmf()
        
        print("Fitting LSA...")
        results['lsa'] = self._fit_lsa()
        
        if GENSIM_AVAILABLE:
            print("Fitting LDA Multicore...")
            results['lda_gensim'] = self._fit_lda_gensim(documents)
        
        # Try advanced models if available
        try:
            print("Fitting BERTopic...")
            results['bertopic'] = self._fit_bertopic(documents)
        except ImportError:
            print("BERTopic not available")
        
        try:
            print("Fitting Top2Vec...")
            results['top2vec'] = self._fit_top2vec(documents)
        except ImportError:
            print("Top2Vec not available")
        
        # Compare models
        results['comparison'] = self._compare_models(documents)
        
        return results
    
    def _fit_lda(self, documents: List[str]) -> Dict:
        """Fit Latent Dirichlet Allocation."""
        
        # Use CountVectorizer for LDA
        count_vectorizer = CountVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            max_df=0.95,
            min_df=2
        )
        count_matrix = count_vectorizer.fit_transform(documents)
        
        lda = LatentDirichletAllocation(
            n_components=self.n_topics,
            random_state=self.random_state,
            learning_method='batch',
            max_iter=50
        )
        
        lda_topics = lda.fit_transform(count_matrix)
        
        # Extract topics
        feature_names = count_vectorizer.get_feature_names_out()
        topics = self._extract_topics(lda, feature_names)
        
        self.models['lda'] = lda
        
        return {
            'model': lda,
            'topics': topics,
            'document_topics': lda_topics,
            'perplexity': lda.perplexity(count_matrix),
            'log_likelihood': lda.score(count_matrix)
        }
    
    def _fit_nmf(self) -> Dict:
        """Fit Non-negative Matrix Factorization."""
        
        nmf = NMF(
            n_components=self.n_topics,
            random_state=self.random_state,
            init='nndsvd',
            max_iter=200
        )
        
        nmf_topics = nmf.fit_transform(self.doc_term_matrix)
        
        # Extract topics
        feature_names = self.vectorizer.get_feature_names_out()
        topics = self._extract_topics(nmf, feature_names)
        
        self.models['nmf'] = nmf
        
        return {
            'model': nmf,
            'topics': topics,
            'document_topics': nmf_topics,
            'reconstruction_error': nmf.reconstruction_err_
        }
    
    def _fit_lsa(self) -> Dict:
        """Fit Latent Semantic Analysis."""
        
        lsa = TruncatedSVD(
            n_components=self.n_topics,
            random_state=self.random_state
        )
        
        lsa_topics = lsa.fit_transform(self.doc_term_matrix)
        
        # Extract topics
        feature_names = self.vectorizer.get_feature_names_out()
        topics = self._extract_topics(lsa, feature_names)
        
        self.models['lsa'] = lsa
        
        return {
            'model': lsa,
            'topics': topics,
            'document_topics': lsa_topics,
            'explained_variance': lsa.explained_variance_ratio_.sum()
        }
    
    def _fit_lda_gensim(self, documents: List[str]) -> Dict:
        """Fit LDA using Gensim."""
        
        # Tokenize documents
        tokenized_docs = [doc.split() for doc in documents]
        
        # Create dictionary and corpus
        dictionary = Dictionary(tokenized_docs)
        corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]
        
        # Train LDA model
        lda_model = LdaMulticore(
            corpus=corpus,
            id2word=dictionary,
            num_topics=self.n_topics,
            random_state=self.random_state,
            passes=10,
            workers=2
        )
        
        # Calculate coherence
        coherence_model = CoherenceModel(
            model=lda_model,
            texts=tokenized_docs,
            dictionary=dictionary,
            coherence='c_v'
        )
        coherence_score = coherence_model.get_coherence()
        
        # Extract topics
        topics = []
        for idx in range(self.n_topics):
            topic_words = lda_model.show_topic(idx, 10)
            topics.append([(word, float(score)) for word, score in topic_words])
        
        self.models['lda_gensim'] = lda_model
        
        return {
            'model': lda_model,
            'topics': topics,
            'coherence': coherence_score,
            'dictionary': dictionary,
            'corpus': corpus
        }
    
    def _fit_bertopic(self, documents: List[str]) -> Dict:
        """Fit BERTopic model."""
        
        from bertopic import BERTopic
        from sentence_transformers import SentenceTransformer
        from umap import UMAP
        from hdbscan import HDBSCAN
        
        # Custom embeddings
        sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
        
        # Custom dimensionality reduction
        umap_model = UMAP(
            n_neighbors=15,
            n_components=5,
            min_dist=0.0,
            metric='cosine',
            random_state=self.random_state
        )
        
        # Custom clustering
        hdbscan_model = HDBSCAN(
            min_cluster_size=15,
            metric='euclidean',
            cluster_selection_method='eom',
            prediction_data=True
        )
        
        # Create BERTopic instance
        topic_model = BERTopic(
            embedding_model=sentence_model,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            nr_topics=self.n_topics,
            calculate_probabilities=True
        )
        
        topics, probs = topic_model.fit_transform(documents)
        
        self.models['bertopic'] = topic_model
        
        return {
            'model': topic_model,
            'topics': topics,
            'probabilities': probs,
            'topic_info': topic_model.get_topic_info(),
            'topic_words': [topic_model.get_topic(i) for i in range(-1, len(set(topics)))]
        }
    
    def _fit_top2vec(self, documents: List[str]) -> Dict:
        """Fit Top2Vec model."""
        
        from top2vec import Top2Vec
        
        model = Top2Vec(
            documents,
            speed="fast-learn",
            workers=4
        )
        
        # Get topic information
        topic_words, word_scores, topic_nums = model.get_topics(self.n_topics)
        
        self.models['top2vec'] = model
        
        return {
            'model': model,
            'num_topics': model.get_num_topics(),
            'topic_words': topic_words,
            'word_scores': word_scores,
            'topic_sizes': model.get_topic_sizes()
        }
    
    def _extract_topics(self, model, feature_names, n_words: int = 10) -> List:
        """Extract top words for each topic."""
        
        topics = []
        for topic_idx, topic in enumerate(model.components_):
            top_words_idx = topic.argsort()[-n_words:][::-1]
            top_words = [(feature_names[i], topic[i]) for i in top_words_idx]
            topics.append(top_words)
        
        return topics
    
    def _compare_models(self, documents: List[str]) -> Dict:
        """Compare different topic models."""
        
        comparison = {}
        
        # Calculate metrics for each model
        for model_name, model_obj in self.models.items():
            if model_name == 'lda':
                comparison[model_name] = {
                    'type': 'Latent Dirichlet Allocation',
                    'n_topics': self.n_topics
                }
            elif model_name == 'nmf':
                comparison[model_name] = {
                    'type': 'Non-negative Matrix Factorization',
                    'n_topics': self.n_topics
                }
            elif model_name == 'lsa':
                comparison[model_name] = {
                    'type': 'Latent Semantic Analysis',
                    'n_topics': self.n_topics
                }
        
        return comparison
    
    def visualize_topics(self, model_name: str = 'lda') -> None:
        """Visualize topics."""
        
        if model_name not in self.models:
            print(f"Model {model_name} not found")
            return
        
        # Create visualization based on model type
        if model_name == 'bertopic' and 'bertopic' in self.models:
            # BERTopic has built-in visualizations
            fig = self.models['bertopic'].visualize_topics()
            return fig
        else:
            # Generic visualization for other models
            self._generic_topic_visualization(model_name)
    
    def _generic_topic_visualization(self, model_name: str) -> None:
        """Generic topic visualization."""
        
        # This would contain visualization code
        print(f"Visualizing {model_name} topics...")
        # Implementation would go here

## 3. Text Classification Pipeline

In [ ]:
class TextClassificationPipeline:
    """Comprehensive text classification pipeline."""
    
    def __init__(self, task: str = 'sentiment'):
        self.task = task
        self.models = {}
        self.vectorizers = {}
        self.label_encoder = LabelEncoder()
        self.best_model = None
        
    def train_multiple_models(self, X_train: List[str], y_train: List[str],
                            X_val: List[str], y_val: List[str]) -> Dict:
        """Train multiple classification models."""
        
        # Encode labels
        y_train_encoded = self.label_encoder.fit_transform(y_train)
        y_val_encoded = self.label_encoder.transform(y_val)
        
        results = {}
        
        # Try different vectorizers
        vectorizers = {
            'tfidf': TfidfVectorizer(max_features=5000, ngram_range=(1, 3)),
            'count': CountVectorizer(max_features=5000, ngram_range=(1, 3)),
            'hashing': HashingVectorizer(n_features=5000, ngram_range=(1, 3))
        }
        
        # Try different models
        classifiers = {
            'naive_bayes': MultinomialNB(),
            'svm': SVC(kernel='linear', probability=True),
            'random_forest': RandomForestClassifier(n_estimators=100, random_state=42)
        }
        
        best_score = 0
        
        for vec_name, vectorizer in vectorizers.items():
            # Vectorize data
            X_train_vec = vectorizer.fit_transform(X_train)
            X_val_vec = vectorizer.transform(X_val)
            
            for clf_name, classifier in classifiers.items():
                model_name = f"{vec_name}_{clf_name}"
                print(f"Training {model_name}...")
                
                # Train model
                classifier.fit(X_train_vec, y_train_encoded)
                
                # Evaluate
                y_pred = classifier.predict(X_val_vec)
                accuracy = accuracy_score(y_val_encoded, y_pred)
                precision, recall, f1, _ = precision_recall_fscore_support(
                    y_val_encoded, y_pred, average='weighted'
                )
                
                results[model_name] = {
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1_score': f1
                }
                
                # Save best model
                if accuracy > best_score:
                    best_score = accuracy
                    self.best_model = classifier
                    self.vectorizers['best'] = vectorizer
                    results['best_model'] = model_name
        
        # Try transformer models if available
        if TRANSFORMERS_AVAILABLE:
            results['transformer'] = self._train_transformer(X_train, y_train, X_val, y_val)
        
        return results
    
    def _train_transformer(self, X_train: List[str], y_train: List[str],
                          X_val: List[str], y_val: List[str]) -> Dict:
        """Train transformer-based classifier."""
        
        from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
        from transformers import AutoTokenizer
        import torch
        from torch.utils.data import Dataset
        
        class TextDataset(Dataset):
            def __init__(self, texts, labels, tokenizer, max_length=128):
                self.texts = texts
                self.labels = labels
                self.tokenizer = tokenizer
                self.max_length = max_length
            
            def __len__(self):
                return len(self.texts)
            
            def __getitem__(self, idx):
                text = self.texts[idx]
                label = self.labels[idx]
                
                encoding = self.tokenizer(
                    text,
                    truncation=True,
                    padding='max_length',
                    max_length=self.max_length,
                    return_tensors='pt'
                )
                
                return {
                    'input_ids': encoding['input_ids'].flatten(),
                    'attention_mask': encoding['attention_mask'].flatten(),
                    'labels': torch.tensor(label, dtype=torch.long)
                }
        
        # Setup transformer
        model_name = 'distilbert-base-uncased'
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        # Encode labels
        y_train_encoded = self.label_encoder.fit_transform(y_train)
        y_val_encoded = self.label_encoder.transform(y_val)
        
        # Create datasets
        train_dataset = TextDataset(X_train, y_train_encoded, tokenizer)
        val_dataset = TextDataset(X_val, y_val_encoded, tokenizer)
        
        # Load model
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=len(self.label_encoder.classes_)
        )
        
        # Training arguments
        training_args = TrainingArguments(
            output_dir='./transformer_classifier',
            num_train_epochs=3,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            warmup_steps=500,
            weight_decay=0.01,
            logging_dir='./logs',
            evaluation_strategy='epoch'
        )
        
        # Create trainer
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset
        )
        
        # Train
        trainer.train()
        
        # Evaluate
        eval_results = trainer.evaluate()
        
        return {
            'model_name': model_name,
            'eval_loss': eval_results['eval_loss'],
            'training_completed': True
        }
    
    def predict(self, texts: List[str]) -> List[str]:
        """Predict labels for new texts."""
        
        if self.best_model is None:
            raise ValueError("No model trained yet")
        
        # Vectorize
        X_vec = self.vectorizers['best'].transform(texts)
        
        # Predict
        y_pred = self.best_model.predict(X_vec)
        
        # Decode labels
        return self.label_encoder.inverse_transform(y_pred)

## 4. Information Extraction and NER

In [ ]:
class InformationExtraction:
    """Advanced information extraction and NER."""
    
    def __init__(self):
        self.nlp = spacy.load('en_core_web_sm')
        
        # Add custom entity patterns if needed
        self.custom_patterns = []
        
        # Initialize transformer NER if available
        if TRANSFORMERS_AVAILABLE:
            self.ner_pipeline = pipeline(
                'ner',
                model='dbmdz/bert-large-cased-finetuned-conll03-english',
                aggregation_strategy='simple'
            )
    
    def extract_all_information(self, text: str) -> Dict:
        """Extract all types of information from text."""
        
        doc = self.nlp(text)
        
        return {
            'entities': self._extract_entities(doc, text),
            'relations': self._extract_relations(doc),
            'events': self._extract_events(doc),
            'temporal': self._extract_temporal(doc),
            'quantities': self._extract_quantities(doc),
            'contact_info': self._extract_contact_info(text),
            'knowledge_graph': self._build_knowledge_graph(doc)
        }
    
    def _extract_entities(self, doc, text: str) -> Dict:
        """Extract named entities using multiple methods."""
        
        entities = {'spacy': {}, 'transformer': {}, 'custom': {}}
        
        # SpaCy NER
        for ent in doc.ents:
            if ent.label_ not in entities['spacy']:
                entities['spacy'][ent.label_] = []
            entities['spacy'][ent.label_].append({
                'text': ent.text,
                'start': ent.start_char,
                'end': ent.end_char
            })
        
        # Transformer NER
        if TRANSFORMERS_AVAILABLE and len(text) < 512:
            try:
                transformer_ents = self.ner_pipeline(text)
                for ent in transformer_ents:
                    label = ent['entity_group']
                    if label not in entities['transformer']:
                        entities['transformer'][label] = []
                    entities['transformer'][label].append({
                        'text': ent['word'],
                        'score': ent['score'],
                        'start': ent['start'],
                        'end': ent['end']
                    })
            except:
                pass
        
        # Custom pattern matching
        entities['custom'] = self._custom_entity_extraction(text)
        
        return entities
    
    def _custom_entity_extraction(self, text: str) -> Dict:
        """Extract entities using custom patterns."""
        
        custom_entities = {}
        
        # Email addresses
        email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
        emails = re.findall(email_pattern, text)
        if emails:
            custom_entities['EMAIL'] = emails
        
        # Phone numbers
        phone_pattern = r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b'
        phones = re.findall(phone_pattern, text)
        if phones:
            custom_entities['PHONE'] = phones
        
        # URLs
        url_pattern = r'https?://(?:[-\w.]|(?:%[\da-fA-F]{2}))+'
        urls = re.findall(url_pattern, text)
        if urls:
            custom_entities['URL'] = urls
        
        # Social Security Numbers (masked)
        ssn_pattern = r'\b\d{3}-\d{2}-\d{4}\b'
        ssns = re.findall(ssn_pattern, text)
        if ssns:
            custom_entities['SSN'] = ['XXX-XX-' + ssn[-4:] for ssn in ssns]
        
        # Credit card numbers (masked)
        cc_pattern = r'\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b'
        ccs = re.findall(cc_pattern, text)
        if ccs:
            custom_entities['CREDIT_CARD'] = ['XXXX-XXXX-XXXX-' + cc[-4:] for cc in ccs]
        
        # IP addresses
        ip_pattern = r'\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b'
        ips = re.findall(ip_pattern, text)
        if ips:
            custom_entities['IP_ADDRESS'] = ips
        
        return custom_entities
    
    def _extract_relations(self, doc) -> List[Dict]:
        """Extract relations between entities."""
        
        relations = []
        
        for sent in doc.sents:
            # Find subject-verb-object triples
            for token in sent:
                if token.dep_ == 'ROOT':
                    subjects = [child for child in token.children if 'subj' in child.dep_]
                    objects = [child for child in token.children if 'obj' in child.dep_]
                    
                    for subj in subjects:
                        for obj in objects:
                            # Check if subject and object are entities
                            subj_ent = None
                            obj_ent = None
                            
                            for ent in sent.ents:
                                if subj.idx >= ent.start_char and subj.idx <= ent.end_char:
                                    subj_ent = ent
                                if obj.idx >= ent.start_char and obj.idx <= ent.end_char:
                                    obj_ent = ent
                            
                            if subj_ent and obj_ent:
                                relations.append({
                                    'subject': subj_ent.text,
                                    'subject_type': subj_ent.label_,
                                    'relation': token.text,
                                    'object': obj_ent.text,
                                    'object_type': obj_ent.label_
                                })
        
        return relations
    
    def _extract_events(self, doc) -> List[Dict]:
        """Extract events from text."""
        
        events = []
        
        for sent in doc.sents:
            # Look for verb-based events
            for token in sent:
                if token.pos_ == 'VERB':
                    event = {
                        'action': token.lemma_,
                        'tense': token.tag_,
                        'participants': [],
                        'location': None,
                        'time': None
                    }
                    
                    # Find participants
                    for child in token.children:
                        if child.dep_ in ['nsubj', 'dobj', 'iobj']:
                            event['participants'].append({
                                'role': child.dep_,
                                'text': child.text
                            })
                    
                    # Find temporal and location info
                    for ent in sent.ents:
                        if ent.label_ in ['DATE', 'TIME']:
                            event['time'] = ent.text
                        elif ent.label_ in ['LOC', 'GPE']:
                            event['location'] = ent.text
                    
                    if event['participants']:
                        events.append(event)
        
        return events
    
    def _extract_temporal(self, doc) -> List[Dict]:
        """Extract temporal information."""
        
        temporal_info = []
        
        for ent in doc.ents:
            if ent.label_ in ['DATE', 'TIME']:
                temporal_info.append({
                    'text': ent.text,
                    'type': ent.label_,
                    'context': ent.sent.text
                })
        
        return temporal_info
    
    def _extract_quantities(self, doc) -> List[Dict]:
        """Extract quantities and measurements."""
        
        quantities = []
        
        for ent in doc.ents:
            if ent.label_ in ['MONEY', 'PERCENT', 'QUANTITY', 'CARDINAL']:
                quantities.append({
                    'text': ent.text,
                    'type': ent.label_,
                    'context': ent.sent.text[:100]
                })
        
        return quantities
    
    def _extract_contact_info(self, text: str) -> Dict:
        """Extract contact information."""
        
        contact_info = {}
        
        # Already extracted in custom entities
        custom_ents = self._custom_entity_extraction(text)
        
        if 'EMAIL' in custom_ents:
            contact_info['emails'] = custom_ents['EMAIL']
        if 'PHONE' in custom_ents:
            contact_info['phones'] = custom_ents['PHONE']
        if 'URL' in custom_ents:
            contact_info['urls'] = custom_ents['URL']
        
        return contact_info
    
    def _build_knowledge_graph(self, doc) -> Dict:
        """Build a simple knowledge graph from the text."""
        
        nodes = []
        edges = []
        
        # Add entities as nodes
        for ent in doc.ents:
            nodes.append({
                'id': f"ent_{ent.start}",
                'label': ent.text,
                'type': ent.label_
            })
        
        # Add relations as edges
        relations = self._extract_relations(doc)
        for i, rel in enumerate(relations):
            edges.append({
                'source': rel['subject'],
                'target': rel['object'],
                'relation': rel['relation']
            })
        
        return {
            'nodes': nodes,
            'edges': edges
        }

## 5. Text Generation and Summarization

In [ ]:
class TextGenerationSummarization:
    """Text generation and summarization capabilities."""
    
    def __init__(self):
        if TRANSFORMERS_AVAILABLE:
            self.summarizer = pipeline('summarization', model='facebook/bart-large-cnn')
            self.generator = pipeline('text-generation', model='gpt2')
            self.qa_pipeline = pipeline('question-answering')
            self.translation_pipeline = pipeline('translation', model='Helsinki-NLP/opus-mt-en-de')
    
    def summarize_text(self, text: str, method: str = 'extractive', 
                      max_length: int = 150, min_length: int = 50) -> str:
        """Summarize text using different methods."""
        
        if method == 'extractive':
            return self._extractive_summarization(text, max_length)
        elif method == 'abstractive' and TRANSFORMERS_AVAILABLE:
            return self._abstractive_summarization(text, max_length, min_length)
        else:
            return self._simple_summarization(text, max_length)
    
    def _extractive_summarization(self, text: str, max_length: int) -> str:
        """Extractive summarization using sentence scoring."""
        
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity
        import networkx as nx
        
        sentences = sent_tokenize(text)
        if len(sentences) < 3:
            return text
        
        # Create TF-IDF matrix
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(sentences)
        
        # Calculate similarity matrix
        similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
        
        # Build graph and rank sentences
        nx_graph = nx.from_numpy_array(similarity_matrix)
        scores = nx.pagerank(nx_graph)
        
        # Sort sentences by score
        ranked_sentences = sorted(
            ((scores[i], s) for i, s in enumerate(sentences)),
            reverse=True
        )
        
        # Select top sentences
        summary_sentences = []
        current_length = 0
        
        for score, sentence in ranked_sentences:
            if current_length + len(sentence.split()) <= max_length:
                summary_sentences.append(sentence)
                current_length += len(sentence.split())
            if current_length >= max_length:
                break
        
        return ' '.join(summary_sentences)
    
    def _abstractive_summarization(self, text: str, max_length: int, min_length: int) -> str:
        """Abstractive summarization using transformers."""
        
        # Truncate text if too long
        if len(text.split()) > 1024:
            text = ' '.join(text.split()[:1024])
        
        summary = self.summarizer(
            text,
            max_length=max_length,
            min_length=min_length,
            do_sample=False
        )
        
        return summary[0]['summary_text']
    
    def _simple_summarization(self, text: str, max_length: int) -> str:
        """Simple summarization based on sentence importance."""
        
        sentences = sent_tokenize(text)
        if len(sentences) <= 3:
            return text
        
        # Take first and last sentences and some from middle
        n_sentences = min(3, len(sentences))
        summary_sentences = [
            sentences[0],
            sentences[len(sentences)//2],
            sentences[-1]
        ]
        
        return ' '.join(summary_sentences[:max_length])
    
    def generate_text(self, prompt: str, max_length: int = 100,
                     temperature: float = 0.7, method: str = 'transformer') -> str:
        """Generate text based on prompt."""
        
        if method == 'transformer' and TRANSFORMERS_AVAILABLE:
            return self._transformer_generation(prompt, max_length, temperature)
        else:
            return self._template_generation(prompt)
    
    def _transformer_generation(self, prompt: str, max_length: int, temperature: float) -> str:
        """Generate text using transformer models."""
        
        generated = self.generator(
            prompt,
            max_length=max_length,
            temperature=temperature,
            num_return_sequences=1
        )
        
        return generated[0]['generated_text']
    
    def _template_generation(self, prompt: str) -> str:
        """Simple template-based generation."""
        
        templates = [
            "Based on '{prompt}', we can say that...",
            "Regarding '{prompt}', it's important to note that...",
            "The topic of '{prompt}' is interesting because..."
        ]
        
        import random
        template = random.choice(templates)
        return template.format(prompt=prompt)
    
    def answer_question(self, question: str, context: str) -> str:
        """Answer questions based on context."""
        
        if TRANSFORMERS_AVAILABLE:
            result = self.qa_pipeline(
                question=question,
                context=context
            )
            return result['answer']
        else:
            # Simple keyword matching
            sentences = sent_tokenize(context)
            question_words = set(word_tokenize(question.lower()))
            
            best_sentence = ""
            best_score = 0
            
            for sentence in sentences:
                sentence_words = set(word_tokenize(sentence.lower()))
                score = len(question_words.intersection(sentence_words))
                
                if score > best_score:
                    best_score = score
                    best_sentence = sentence
            
            return best_sentence if best_sentence else "Answer not found in context."

## 6. Example Usage and Demonstrations

In [ ]:
# Example 1: Comprehensive Text Analysis
def demo_text_analysis():
    """Demonstrate text analysis capabilities."""
    
    sample_text = """
    Apple Inc. announced its quarterly earnings on January 15, 2024, reporting a revenue of $123.9 billion,
    a 5% increase from the previous quarter. CEO Tim Cook stated that the company's strong performance
    was driven by iPhone sales in China and the successful launch of the Vision Pro headset.
    The company also announced a $110 billion stock buyback program and increased its dividend by 4%.
    Investors reacted positively to the news, with Apple's stock price rising 3.2% in after-hours trading.
    """
    
    analyzer = ComprehensiveTextAnalyzer()
    results = analyzer.analyze_text(sample_text)
    
    print("Text Analysis Results:")
    print("=" * 50)
    
    # Basic statistics
    print("\nBasic Statistics:")
    for key, value in results['basic_stats'].items():
        print(f"  {key}: {value:.2f}" if isinstance(value, float) else f"  {key}: {value}")
    
    # Sentiment
    print("\nSentiment Analysis:")
    if 'vader' in results['sentiment']:
        print(f"  VADER Compound Score: {results['sentiment']['vader']['compound']:.3f}")
    
    # Entities
    print("\nNamed Entities:")
    for entity_type, entities in results['entities']['entities'].items():
        print(f"  {entity_type}: {', '.join(entities[:3])}")
    
    # Key phrases
    print("\nKey Phrases:")
    if 'noun_phrases' in results['key_phrases']:
        top_phrases = results['key_phrases']['noun_phrases'][:5]
        for phrase, count in top_phrases:
            print(f"  - {phrase} ({count} occurrences)")
    
    return results

# Example 2: Topic Modeling
def demo_topic_modeling():
    """Demonstrate topic modeling."""
    
    # Sample documents
    documents = [
        "Machine learning is transforming the way we analyze data and make predictions.",
        "Deep neural networks have revolutionized computer vision and image recognition.",
        "Natural language processing enables computers to understand human language.",
        "Climate change is one of the most pressing challenges of our time.",
        "Renewable energy sources like solar and wind are becoming more cost-effective.",
        "The stock market showed strong gains in the technology sector.",
        "Artificial intelligence is being integrated into various industries.",
        "Environmental sustainability is crucial for future generations.",
        "Financial markets are influenced by global economic trends.",
        "Data science combines statistics, programming, and domain expertise."
    ]
    
    topic_modeler = AdvancedTopicModeling(n_topics=3)
    results = topic_modeler.fit_all_models(documents)
    
    print("\nTopic Modeling Results:")
    print("=" * 50)
    
    # Display LDA topics
    if 'lda' in results:
        print("\nLDA Topics:")
        for i, topic in enumerate(results['lda']['topics']):
            top_words = ', '.join([word for word, _ in topic[:5]])
            print(f"  Topic {i+1}: {top_words}")
    
    return results

# Example 3: Information Extraction
def demo_information_extraction():
    """Demonstrate information extraction."""
    
    sample_text = """
    John Smith, CEO of Tech Innovations Inc., announced a partnership with Global Systems Ltd.
    on March 15, 2024. The deal, worth $45 million, will combine Tech Innovations' AI expertise
    with Global Systems' cloud infrastructure. For more information, contact press@techinnovations.com
    or call 555-123-4567. Visit our website at https://www.techinnovations.com.
    """
    
    extractor = InformationExtraction()
    results = extractor.extract_all_information(sample_text)
    
    print("\nInformation Extraction Results:")
    print("=" * 50)
    
    # Entities
    print("\nExtracted Entities:")
    for source, entities in results['entities'].items():
        if entities:
            print(f"\n  {source.upper()}:")
            for entity_type, items in entities.items():
                if isinstance(items, list) and items:
                    print(f"    {entity_type}: {items}")
    
    # Contact Information
    if results['contact_info']:
        print("\nContact Information:")
        for info_type, info in results['contact_info'].items():
            print(f"  {info_type}: {info}")
    
    # Relations
    if results['relations']:
        print("\nExtracted Relations:")
        for relation in results['relations'][:3]:
            print(f"  {relation['subject']} --{relation['relation']}--> {relation['object']}")
    
    return results

# Run demonstrations
if __name__ == "__main__":
    print("Running NLP Pipeline Demonstrations...\n")
    
    # Run text analysis demo
    analysis_results = demo_text_analysis()
    
    # Run topic modeling demo
    topic_results = demo_topic_modeling()
    
    # Run information extraction demo
    extraction_results = demo_information_extraction()
    
    print("\nAll demonstrations completed successfully!")

## Summary

This comprehensive NLP pipeline provides:

1. **Text Analysis**: Complete linguistic analysis including readability, sentiment, POS, and complexity metrics
2. **Topic Modeling**: Multiple algorithms (LDA, NMF, LSA, BERTopic) with comparison
3. **Text Classification**: Traditional ML and transformer-based classification
4. **Information Extraction**: NER, relation extraction, event detection, knowledge graphs
5. **Text Generation**: Summarization (extractive/abstractive), text generation, Q&A
6. **Multi-lingual Support**: Extensible to multiple languages
7. **Production Ready**: Modular, configurable, and scalable design

The pipeline can be easily extended and customized for specific NLP tasks and domains.